In [308]:
import pandas as pd 

In [309]:
df_emails = pd.read_csv('../../dataset/original/emails.csv')
print(df_emails.info)

<bound method DataFrame.info of         Co_Ref Time_to_Renewal crm_accreditation_completed  \
0       KG5766     pre_renewal               Not Discussed   
1       EJ1532          14_out               Not Discussed   
2       AA4063      prior_year               Not Discussed   
3       JY9888      prior_year                          No   
4       WO6689     pre_renewal               Not Discussed   
...        ...             ...                         ...   
123384  LY8301      prior_year               Not Discussed   
123385  ME4112      prior_year               Not Discussed   
123386  QB0417      prior_year               Not Discussed   
123387  SM3404      prior_year               Not Discussed   
123388  YD6212      prior_year               Not Discussed   

       crm_timely_completion crm_progress_towards_accreditation  \
0              Not Discussed                      Not Discussed   
1              Not Discussed                      Not Discussed   
2              Not Dis

C:\Users\Asus\AppData\Local\Temp\ipykernel_31040\4293998071.py:1: DtypeWarning: Columns (16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_emails = pd.read_csv('../../dataset/original/emails.csv')


## Handling duplicate cols 

In [310]:
print( "Before duplication removal:",df_emails.duplicated().sum())
df_emails = df_emails.drop_duplicates()
print( "After duplication removal:",df_emails.duplicated().sum())

print("Total rows after duplication removal:", len(df_emails))


Before duplication removal: 0
After duplication removal: 0
Total rows after duplication removal: 123389


## Fixing Datatypes and standardising

In [311]:
df_validation = pd.read_csv("../../dataset/01_raw/emails/data_validation_summary.csv")
print(df_validation.info)

<bound method DataFrame.info of     Unnamed: 0                           column_name data_type  column_type  \
0            0                                Co_Ref    object  categorical   
1            1                       Time_to_Renewal    object  categorical   
2            2           crm_accreditation_completed    object  categorical   
3            3                 crm_timely_completion    object  categorical   
4            4    crm_progress_towards_accreditation    object  categorical   
5            5           crm_delays_in_accreditation    object  categorical   
6            6        crm_contractor_suggested_leave    object  categorical   
7            7             crm_contractor_engagement    object  categorical   
8            8              crm_contractor_sentiment    object  categorical   
9            9        crm_contractor_sentiment_score    object  categorical   
10          10             crm_dts_or_ssip_mentioned    object  categorical   
11          11      

In [312]:
df_mixed = pd.read_csv("../../dataset/01_raw/emails/mixed_data_types.csv")
print(df_mixed)

                                 column  numeric_count  non_numeric_count  \
0             crm_contractor_engagement              1             102353   
1        crm_contractor_sentiment_score          71159              31195   
2             crm_dts_or_ssip_mentioned             20             102334   
3        crm_customer_payment_intention              4             102350   
4                 crm_agent_chase_count         111696                538   
5                crm_membership_overdue              1             112233   
6               crm_auto_renewal_status         112185                 49   
7  crm_dissatisified_with_renewal_price             40             112194   

   sample_numeric                                sample_non_numeric  
0               0                                               Yes  
1              50                                     Not Discussed  
2              20                                                No  
3              20         

In [313]:
df_emails['crm_contractor_engagement'].unique()

array(['Yes', 'No', 'Not Discussed', nan, ' the price is too high',
       ' indicating dissatisfaction with the service or support provided."',
       ' indicating dissatisfaction with the cost."',
       ' which they claim to have requested last year."',
       '"" indicating dissatisfaction with the service."',
       '"" making it not feasible to renew."',
       '"" implying they do not wish to renew their membership."',
       ' and they have chosen to drop clients that require the certification."',
       ' indicating uncertainty about continuing with the service."',
       '"" which led to a discussion about a one-off saving and the value of the SafeContractor membership."',
       ' citing time constraints and difficulties in responding to multiple emails."',
       ' leading to frustration and a desire to cancel."',
       ' finding the cost and effort ridiculous."',
       ' just asked for invoice""."',
       '"" which led them to consider canceling their accreditation."',


In [314]:
df_emails['crm_contractor_engagement'] = (
    df_emails['crm_contractor_engagement']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [315]:
df_emails['crm_contractor_engagement'] = (
    df_emails['crm_contractor_engagement']
    .replace('nan', 'unknown')
)

In [316]:
def clean_engagement(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    elif 'not discussed' in val:
        return 'no'
    
    # numeric noise
    elif val.isdigit():
        return 'unknown'
    
    # long sentences → unknown (basic cleaning stage)
    elif len(val) > 15:
        return 'unknown'
    
    else:
        return 'unknown'

df_emails['crm_contractor_engagement'] = df_emails['crm_contractor_engagement'].apply(clean_engagement)

In [317]:
df_emails['crm_contractor_engagement'].unique()

array(['yes', 'no', 'unknown'], dtype=object)

In [318]:
df_emails['crm_contractor_sentiment_score'].unique()

array(['50', 'Not Discussed', '80', '0', '40', '30', '20', '100', '31',
       '90', '39', '35', '60', '84', '37', '45', '38', '16', '83', '91',
       '82', '99', '95', '46', '97', '33', '42', '93', '43', '48', nan,
       '70', '27', '25', 'Yes', 'Dissatisfied', '36', '55', '41', '10',
       'Neutral', 'Satisfied', '49', '94', '98', '86', '85', '25.5',
       '57.5', '62.5', '87', '96', '27.5'], dtype=object)

In [319]:
col = 'crm_contractor_sentiment_score'

df_emails[col] = (
    df_emails[col]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [320]:
def map_sentiment(val):
    if val == 'satisfied':
        return 80
    elif val == 'neutral':
        return 50
    elif val == 'dissatisfied':
        return 20
    elif val == 'not discussed':
        return None
    elif val == 'yes':   # noise
        return None
    else:
        return val

df_emails[col] = df_emails[col].apply(map_sentiment)

In [321]:
df_emails['crm_contractor_sentiment_score'].unique()

array(['50', None, '80', '0', '40', '30', '20', '100', '31', '90', '39',
       '35', '60', '84', '37', '45', '38', '16', '83', '91', '82', '99',
       '95', '46', '97', '33', '42', '93', '43', '48', 'nan', '70', '27',
       '25', 20, '36', '55', '41', '10', 50, 80, '49', '94', '98', '86',
       '85', '25.5', '57.5', '62.5', '87', '96', '27.5'], dtype=object)

In [322]:
df_emails['crm_dts_or_ssip_mentioned'].unique()

array(['No', 'Yes', nan, 'Dissatisfied', '20', '30', '0', '50', '80',
       'Not Discussed'], dtype=object)

In [323]:
df_emails['crm_dts_or_ssip_mentioned'] = (
    df_emails['crm_dts_or_ssip_mentioned']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [324]:
df_emails['crm_dts_or_ssip_mentioned'] = (
    df_emails['crm_dts_or_ssip_mentioned']
    .replace('nan', 'unknown')
)

In [325]:
def clean_dts(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    elif 'not discussed' in val:
        return 'no'
    
    # noise (numbers or wrong labels)
    elif val.isdigit() or 'dissatisfied' in val:
        return 'unknown'
    
    else:
        return 'unknown'

df_emails['crm_dts_or_ssip_mentioned'] = df_emails['crm_dts_or_ssip_mentioned'].apply(clean_dts)

In [326]:
df_emails['crm_dts_or_ssip_mentioned'].unique()

array(['no', 'yes', 'unknown'], dtype=object)

In [327]:
df_emails['crm_customer_payment_intention'].unique()

array(['No', 'Not Discussed', 'Yes', nan, '20', '0', '30'], dtype=object)

In [328]:
df_emails['crm_customer_payment_intention'] = (
    df_emails['crm_customer_payment_intention']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [329]:
df_emails['crm_customer_payment_intention'] = (
    df_emails['crm_customer_payment_intention']
    .replace('nan', 'unknown')
)

In [330]:
def clean_payment(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    elif 'not discussed' in val:
        return 'unknown'   # safer than forcing 'no'
    
    # numeric noise
    elif val.isdigit():
        return 'unknown'
    
    else:
        return 'unknown'

df_emails['crm_customer_payment_intention'] =  df_emails['crm_customer_payment_intention'].apply(clean_payment)

In [331]:
df_emails['crm_customer_payment_intention'].unique()

array(['no', 'unknown', 'yes'], dtype=object)

In [332]:

df_emails['crm_agent_chase_count'].unique()

array(['1', '0', '3', '2', '4', '7', '9', '6', '5', '11', 'More than 5',
       '14', '8', 'Multiple', 'Many', 'A few',
       'Multiple ( exact number not specified)', 'Not specified', '10',
       'Several', 'Multiple times ( exact number not specified)', '19',
       '05-Jun', 'Lots (no specific number mentioned)',
       'A number of times (no specific number mentioned)',
       'Quite a few times (no exact number provided)', 'Every month',
       'Not explicitly stated, but implied to be multiple times', '13',
       nan, 'Not Discussed',
       'Not applicable (only one attempt to contact)',
       'Not applicable (only one email)', 'Multiple times (at least 7-8)',
       'Multiple times weekly', 'Multiple (exact number not specified)',
       '2 (at least, as the agent mentioned sending emails and making a call)',
       '17', '10+', 'Multiple times (exact number not specified)',
       'Multiple times (at least 7 emails and several phone calls)',
       'Multiple times (implied

In [333]:
col = 'crm_agent_chase_count'

df_emails[col] = (
    df_emails[col]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [334]:
def clean_chase(val):
    
    # direct numbers
    if val.isdigit():
        return int(val)
    
    # ranges / plus
    elif '+' in val:
        return int(val.replace('+','')) + 1
    
    elif 'more than' in val:
        num = ''.join(filter(str.isdigit, val))
        return int(num) + 1 if num else None
    
    # text approximations
    elif 'few' in val:
        return 3
    elif 'several' in val:
        return 4
    elif 'many' in val or 'multiple' in val or 'numerous' in val:
        return 6
    
    # monthly / repeated patterns
    elif 'monthly' in val or 'every month' in val:
        return 6
    
    # not applicable / not discussed
    elif 'not' in val:
        return 0
    
    # everything else
    else:
        return None


df_emails[col] = df_emails[col].apply(clean_chase)
df_emails[col] = pd.to_numeric(df_emails[col], errors='coerce')


In [335]:

df_emails['crm_agent_chase_count'].unique()

array([ 1.,  0.,  3.,  2.,  4.,  7.,  9.,  6.,  5., 11., 14.,  8., 10.,
       19., nan, 13., 17., 15., 21., 12., 24.])

In [336]:
df_emails['crm_membership_overdue'].unique()

array(['Yes', 'Not Discussed', 'No',
       ' as a ""Deem to Satisfy"" for their CHAS accreditation',
       ' despite being up to date."',
       ' which they requested to be removed and the invoice reissued."',
       ' and the manner in which it was executed."',
       ' preventing them from renewing their membership."',
       ' which is not currently included."',
       ' possibly because it was too early."',
       ' and the invoice amount not being reduced despite not conducting a full audit."',
       ' and also requested clarification on the renewal health and safety assessment."',
       ' which they claimed was damaging to their business."',
       ' indeed too busy""."',
       ' citing that the questions did not apply to him as he works alone."',
       ' leading to frustration and multiple calls to chase the audit."',
       '"" and was unhappy with the level of customer service."',
       ' and also had problems providing site-specific RAMS and evidence of refresher trai

In [337]:
col = 'crm_membership_overdue'

df_emails[col] = (
    df_emails[col]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [338]:
df_emails[col] = df_emails[col].replace('nan', 'unknown')

In [339]:
def clean_overdue(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    
    # semi-valid
    elif 'not discussed' in val:
        return 'unknown'
    elif 'almost expired' in val:
        return 'yes'   # close to overdue
    
    # numeric noise
    elif val.isdigit():
        return 'unknown'
    
    # long text → noise
    elif len(val) > 20:
        return 'unknown'
    
    else:
        return 'unknown'

df_emails[col] = df_emails[col].apply(clean_overdue)

In [340]:
df_emails['crm_membership_overdue'].unique()

array(['yes', 'unknown', 'no'], dtype=object)

In [341]:
df_emails['crm_auto_renewal_status'].unique()

array(['0', '2', '1', ' rendering their current accreditation invalid."',
       'Not Discussed', 'No', 'Yes', ' potentially up to 8 months."',
       ' and over-querying of supplied evidence."', nan,
       ' and not being familiar with it."',
       ' and also had to provide additional information for the accreditation process."',
       ' but was helped to correct this by the agent."',
       ' including evidence of face masks and Respiratory Protective Equipment (RPE)."',
       ' and employer\'s liability."', 0], dtype=object)

In [342]:
col = 'crm_auto_renewal_status'

df_emails[col] = (
    df_emails[col]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [343]:
df_emails[col] = df_emails[col].replace('nan', 'unknown')

In [344]:
def clean_auto(val):
    
    # direct mapping
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    
    # numeric mapping
    elif val == '1':
        return 'yes'
    elif val == '0':
        return 'no'
    elif val == '2':
        return 'unknown'
    
    # not discussed
    elif 'not discussed' in val:
        return 'unknown'
    
    # noise (sentences)
    elif len(val) > 15:
        return 'unknown'
    
    else:
        return 'unknown'

df_emails[col] = df_emails[col].apply(clean_auto)

In [345]:
df_emails['crm_auto_renewal_status'].unique()

array(['no', 'unknown', 'yes'], dtype=object)

In [346]:
df_emails['crm_dissatisified_with_renewal_price'].unique()

array(['No', 'Not Discussed', 'Yes', '0', '2', nan,
       'No actions or steps were taken by the agent in the provided email content.'],
      dtype=object)

In [347]:
col = 'crm_dissatisified_with_renewal_price'

df_emails[col] = (
    df_emails[col]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [348]:
df_emails[col] = df_emails[col].replace('nan', 'unknown')

In [349]:
def clean_price_dissatisfaction(val):
    
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    
    # numeric mapping
    elif val == '0':
        return 'no'
    elif val == '2':
        return 'unknown'
    
    # not discussed
    elif 'not discussed' in val:
        return 'unknown'
    
    # long sentence → noise
    elif len(val) > 20:
        return 'unknown'
    
    else:
        return 'unknown'

df_emails[col] = df_emails[col].apply(clean_price_dissatisfaction)

In [350]:
df_emails['crm_dissatisified_with_renewal_price'].unique()

array(['no', 'unknown', 'yes'], dtype=object)

## Analyzing Cols with more NULLs 

In [351]:
print(len(df_emails))
# print(df_emails.isnull().sum())

print(len(df_emails) - df_emails.isnull().sum())

123389
Co_Ref                                  123389
Time_to_Renewal                         123389
crm_accreditation_completed             102354
crm_timely_completion                   102354
crm_progress_towards_accreditation      102354
crm_delays_in_accreditation             102354
crm_contractor_suggested_leave          102354
crm_contractor_engagement               123389
crm_contractor_sentiment                102354
crm_contractor_sentiment_score           92214
crm_dts_or_ssip_mentioned               123389
crm_customer_payment_intention          123389
crm_competitors_mentioned               112234
crm_membership_level                    112234
crm_platform_issues_raised              112234
crm_agent_chased_contractor             112234
crm_agent_chase_count                   112228
crm_accreditation_issues                112234
crm_membership_overdue                  123389
crm_auto_renewal_status                 123389
crm_dissatisified_with_renewal_price    123389
crm_cu

In [352]:
print(
    df_validation[['column_name', 'column_type', 'null_percentage']]
    .sort_values(by='null_percentage', ascending=False)
)

                             column_name  column_type  null_percentage
9         crm_contractor_sentiment_score  categorical            17.05
2            crm_accreditation_completed  categorical            17.05
3                  crm_timely_completion  categorical            17.05
4     crm_progress_towards_accreditation  categorical            17.05
5            crm_delays_in_accreditation  categorical            17.05
6         crm_contractor_suggested_leave  categorical            17.05
7              crm_contractor_engagement  categorical            17.05
8               crm_contractor_sentiment  categorical            17.05
10             crm_dts_or_ssip_mentioned  categorical            17.05
11        crm_customer_payment_intention  categorical            17.05
25      crm_financial_hardship_mentioned  categorical             9.30
21               crm_customer_complained  categorical             9.30
22                  crm_refund_mentioned  categorical             9.30
23    

In [353]:

cols_to_drop = df_validation[
    df_validation['null_percentage'] > 90
]['column_name'].tolist()


target = 'Churn_Category'   

if target in cols_to_drop:
    cols_to_drop.remove(target)


print("Before dropping columns:", len(df_emails.columns))
print("Columns to drop:", cols_to_drop)


df_emails = df_emails.drop(columns=cols_to_drop, errors='ignore')

print("After dropping columns:", len(df_emails.columns))

Before dropping columns: 27
Columns to drop: []
After dropping columns: 27


## ================================================================

In [354]:
df_emails.to_csv('../../dataset/02_basic_clean/emails.csv', index=False)